# Hyperparameter Optimization (Phase 4)

Two-stage tuning policy for the Top-3 models (from `results/top3_models.csv`), on every dataset and every shared output target:

1. **Random Search** — broad exploration over the registry's search space (`configs/config.yaml: tuning.random_search_iterations`).
2. **Bayesian Optimization (Optuna, TPE sampler)** — focused refinement (`tuning.optuna_trials`).

Grid Search is intentionally not implemented (forbidden for this small-data project). Best CV RMSE and best parameters from both stages are saved to `results/tuning/top3_tuning_results.csv`, and the last cell shows the complete results.

Setup: repo imports, pandas display options, load the Top-3 models and all datasets.

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if (Path.cwd() / '..' / 'src').resolve().exists() else Path.cwd()
SCRIPTS_DIR = REPO_ROOT / 'scripts'
for p in (REPO_ROOT, SCRIPTS_DIR):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

import pandas as pd

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 250)

from _pipeline_common import read_top3_models
from src.config import get_base_seed, get_path, load_config
from src.data.loader import load_all_datasets
from src.tuning.hyperparameter import optuna_tune, random_search
from src.utils.io import save_table

cfg = load_config()
seed = get_base_seed()
top3 = read_top3_models()
bundles = load_all_datasets(discrete=False)
print('Top-3 models to tune:', top3)
print('Random search iterations:', cfg['tuning']['random_search_iterations'])
print('Optuna trials:', cfg['tuning']['optuna_trials'])

Top-3 models to tune: ['xgboost', 'gpr', 'lightgbm']
Random search iterations: 30
Optuna trials: 50


## Tuning on Dataset_0136

Random Search followed by Optuna Bayesian optimization, for every shared target and every Top-3 model.

In [2]:
rows_0136 = []
bundle = bundles['Dataset_0136']
shared_targets = [c for c in cfg['shared_outputs'] if c in bundle.Y.columns]
for target in shared_targets:
    y = bundle.Y[target]
    for model_name in top3:
        rs = random_search(model_name, bundle.X, y, seed=seed)
        study = optuna_tune(model_name, bundle.X, y, seed=seed)
        rows_0136.append({
            'dataset': 'Dataset_0136',
            'target': target,
            'model': model_name,
            'random_search_best_rmse': -rs.best_score_,
            'random_search_best_params': str(rs.best_params_),
            'optuna_best_rmse': -study.best_value,
            'optuna_best_params': str(study.best_params),
        })
        print(f'[tuning] Dataset_0136 / {target} / {model_name} done')

tuning_0136 = pd.DataFrame(rows_0136)
tuning_0136

[tuning] Dataset_0136 / Si Particle Size (um) / xgboost done


C:\Users\mohammadhosein\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\gaussian_process\kernels.py:440: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-06. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(


[tuning] Dataset_0136 / Si Particle Size (um) / gpr done


[tuning] Dataset_0136 / Si Particle Size (um) / lightgbm done


[tuning] Dataset_0136 / Hardness (HV) / xgboost done


...

[tuning] Dataset_0136 / Elongation (%) / xgboost done


C:\Users\mohammadhosein\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\gaussian_process\kernels.py:440: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-06. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(


[tuning] Dataset_0136 / Elongation (%) / gpr done


[tuning] Dataset_0136 / Elongation (%) / lightgbm done


,dataset,target,model,random_search_best_rmse,random_search_best_params,optuna_best_rmse,optuna_best_params
0,Dataset_0136,Si Particle Size (um),xgboost,0.167731,{'model__learning_rate': np.float64(0.03142064...,0.156177,"{'model__n_estimators': 681, 'model__learning_..."
1,Dataset_0136,Si Particle Size (um),gpr,0.259406,{'model__alpha': np.float64(0.0580496585534013...,0.257787,{'model__alpha': 0.07789901985338313}
2,Dataset_0136,Si Particle Size (um),lightgbm,0.182430,{'model__learning_rate': np.float64(0.08124173...,0.184138,"{'model__n_estimators': 297, 'model__learning_..."
3,Dataset_0136,Hardness (HV),xgboost,1.077698,{'model__learning_rate': np.float64(0.07665788...,0.972185,"{'model__n_estimators': 173, 'model__learning_..."
4,Dataset_0136,Hardness (HV),gpr,1.458536,{'model__alpha': np.float64(1.1737996092661853...,1.458536,{'model__alpha': 1.6550213716969952e-05}
5,Dataset_0136,Hardness (HV),lightgbm,0.900878,{'model__learning_rate': np.float64(0.25540068...,0.819172,"{'model__n_estimators': 596, 'model__learning_..."
6,Dataset_0136,wear rate,xgboost,0.200207,{'model__learning_rate': np.float64(0.06416866...,0.177737,"{'model__n_estimators': 715, 'model__learning_..."
7,Dataset_0136,wear rate,gpr,0.290832,{'model__alpha': np.float64(3.197999889429474e...,0.290832,{'model__alpha': 3.284959743501089e-05}
8,Dataset_0136,wear rate,lightgbm,0.198334,{'model__learning_rate': np.float64(0.03385139...,0.206369,"{'model__n_estimators': 583, 'model__learning_..."
9,Dataset_0136,Ultimate Compression Strength (MPa),xgboost,3.933567,{'model__learning_rate': np.float64(0.07665788...,3.721806,"{'model__n_estimators': 697, 'model__learning_..."


## Tuning on Dataset_0172

Random Search followed by Optuna Bayesian optimization, for every shared target and every Top-3 model.

In [3]:
rows_0172 = []
bundle = bundles['Dataset_0172']
shared_targets = [c for c in cfg['shared_outputs'] if c in bundle.Y.columns]
for target in shared_targets:
    y = bundle.Y[target]
    for model_name in top3:
        rs = random_search(model_name, bundle.X, y, seed=seed)
        study = optuna_tune(model_name, bundle.X, y, seed=seed)
        rows_0172.append({
            'dataset': 'Dataset_0172',
            'target': target,
            'model': model_name,
            'random_search_best_rmse': -rs.best_score_,
            'random_search_best_params': str(rs.best_params_),
            'optuna_best_rmse': -study.best_value,
            'optuna_best_params': str(study.best_params),
        })
        print(f'[tuning] Dataset_0172 / {target} / {model_name} done')

tuning_0172 = pd.DataFrame(rows_0172)
tuning_0172

[tuning] Dataset_0172 / Si Particle Size (um) / xgboost done


[tuning] Dataset_0172 / Si Particle Size (um) / gpr done


[tuning] Dataset_0172 / Si Particle Size (um) / lightgbm done


[tuning] Dataset_0172 / Hardness (HV) / xgboost done


C:\Users\mohammadhosein\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\gaussian_process\kernels.py:440: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-06. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(


...

[tuning] Dataset_0172 / Ultimate Compression Strength (MPa) / lightgbm done


[tuning] Dataset_0172 / Elongation (%) / xgboost done


[tuning] Dataset_0172 / Elongation (%) / gpr done


[tuning] Dataset_0172 / Elongation (%) / lightgbm done


,dataset,target,model,random_search_best_rmse,random_search_best_params,optuna_best_rmse,optuna_best_params
0,Dataset_0172,Si Particle Size (um),xgboost,0.101725,{'model__learning_rate': np.float64(0.03142064...,0.104337,"{'model__n_estimators': 659, 'model__learning_..."
1,Dataset_0172,Si Particle Size (um),gpr,0.158276,{'model__alpha': np.float64(3.0900814009365934...,0.158276,{'model__alpha': 0.009516684779883045}
2,Dataset_0172,Si Particle Size (um),lightgbm,0.122095,{'model__learning_rate': np.float64(0.03385139...,0.133400,"{'model__n_estimators': 563, 'model__learning_..."
3,Dataset_0172,Hardness (HV),xgboost,1.141091,{'model__learning_rate': np.float64(0.06416866...,1.097219,"{'model__n_estimators': 603, 'model__learning_..."
4,Dataset_0172,Hardness (HV),gpr,1.705316,{'model__alpha': np.float64(0.0580496585534013...,1.711705,{'model__alpha': 0.0002358594058414263}
5,Dataset_0172,Hardness (HV),lightgbm,1.134611,{'model__learning_rate': np.float64(0.24040287...,1.108512,"{'model__n_estimators': 483, 'model__learning_..."
6,Dataset_0172,wear rate,xgboost,0.145940,{'model__learning_rate': np.float64(0.05306768...,0.143341,"{'model__n_estimators': 782, 'model__learning_..."
7,Dataset_0172,wear rate,gpr,0.220366,{'model__alpha': np.float64(2.936998110437699e...,0.219205,{'model__alpha': 0.022079952071046495}
8,Dataset_0172,wear rate,lightgbm,0.181222,{'model__learning_rate': np.float64(0.21574860...,0.155528,"{'model__n_estimators': 508, 'model__learning_..."
9,Dataset_0172,Ultimate Compression Strength (MPa),xgboost,5.381129,{'model__learning_rate': np.float64(0.05306768...,5.210698,"{'model__n_estimators': 750, 'model__learning_..."


## Tuning on Dataset_3772

Random Search followed by Optuna Bayesian optimization, for every shared target and every Top-3 model.

In [4]:
rows_3772 = []
bundle = bundles['Dataset_3772']
shared_targets = [c for c in cfg['shared_outputs'] if c in bundle.Y.columns]
for target in shared_targets:
    y = bundle.Y[target]
    for model_name in top3:
        rs = random_search(model_name, bundle.X, y, seed=seed)
        study = optuna_tune(model_name, bundle.X, y, seed=seed)
        rows_3772.append({
            'dataset': 'Dataset_3772',
            'target': target,
            'model': model_name,
            'random_search_best_rmse': -rs.best_score_,
            'random_search_best_params': str(rs.best_params_),
            'optuna_best_rmse': -study.best_value,
            'optuna_best_params': str(study.best_params),
        })
        print(f'[tuning] Dataset_3772 / {target} / {model_name} done')

tuning_3772 = pd.DataFrame(rows_3772)
tuning_3772

[tuning] Dataset_3772 / Si Particle Size (um) / xgboost done


[tuning] Dataset_3772 / Si Particle Size (um) / gpr done


[tuning] Dataset_3772 / Si Particle Size (um) / lightgbm done


[tuning] Dataset_3772 / Hardness (HV) / xgboost done


[tuning] Dataset_3772 / Hardness (HV) / gpr done


...

[tuning] Dataset_3772 / Ultimate Compression Strength (MPa) / lightgbm done


[tuning] Dataset_3772 / Elongation (%) / xgboost done


[tuning] Dataset_3772 / Elongation (%) / gpr done


[tuning] Dataset_3772 / Elongation (%) / lightgbm done


,dataset,target,model,random_search_best_rmse,random_search_best_params,optuna_best_rmse,optuna_best_params
0,Dataset_3772,Si Particle Size (um),xgboost,0.046193,{'model__learning_rate': np.float64(0.03142064...,0.042879,"{'model__n_estimators': 245, 'model__learning_..."
1,Dataset_3772,Si Particle Size (um),gpr,0.061981,{'model__alpha': np.float64(4.247270739805821e...,0.065877,{'model__alpha': 1.0197503947628369e-10}
2,Dataset_3772,Si Particle Size (um),lightgbm,0.039589,{'model__learning_rate': np.float64(0.08124173...,0.057674,"{'model__n_estimators': 210, 'model__learning_..."
3,Dataset_3772,Hardness (HV),xgboost,1.504147,{'model__learning_rate': np.float64(0.21130455...,1.526547,"{'model__n_estimators': 307, 'model__learning_..."
4,Dataset_3772,Hardness (HV),gpr,2.233281,{'model__alpha': np.float64(3.676834516546548e...,2.233203,{'model__alpha': 0.015783231288070406}
5,Dataset_3772,Hardness (HV),lightgbm,1.705566,{'model__learning_rate': np.float64(0.05291912...,1.616591,"{'model__n_estimators': 481, 'model__learning_..."
6,Dataset_3772,wear rate,xgboost,0.156916,{'model__learning_rate': np.float64(0.06416866...,0.141213,"{'model__n_estimators': 604, 'model__learning_..."
7,Dataset_3772,wear rate,gpr,0.180917,{'model__alpha': np.float64(3.0900814009365934...,0.180918,{'model__alpha': 2.8321795431027907e-08}
8,Dataset_3772,wear rate,lightgbm,0.158881,{'model__learning_rate': np.float64(0.21574860...,0.161846,"{'model__n_estimators': 301, 'model__learning_..."
9,Dataset_3772,Ultimate Compression Strength (MPa),xgboost,7.034215,{'model__learning_rate': np.float64(0.03142064...,7.266331,"{'model__n_estimators': 756, 'model__learning_..."


## Final Summary — Complete Hyperparameter Tuning Results (All Datasets)

Random Search vs Optuna best RMSE, side by side, saved to `results/tuning/top3_tuning_results.csv`.

In [5]:
tuning_all = pd.concat([tuning_0136, tuning_0172, tuning_3772], ignore_index=True)
save_table(tuning_all, get_path('results_dir') / 'tuning' / 'top3_tuning_results.csv')

print('=' * 90)
print('HYPERPARAMETER TUNING RESULTS — Random Search vs Optuna (all datasets)')
print('=' * 90)
display(tuning_all)

improvement = tuning_all.copy()
improvement['optuna_improves_over_random'] = (
    improvement['optuna_best_rmse'] < improvement['random_search_best_rmse']
)
print()
print('Optuna improved on Random Search in',
      f"{improvement['optuna_improves_over_random'].sum()} / {len(improvement)}",
      'model/target/dataset combinations')

print()
print('Tuning results saved to:', (get_path('results_dir') / 'tuning').resolve())

HYPERPARAMETER TUNING RESULTS — Random Search vs Optuna (all datasets)


,dataset,target,model,random_search_best_rmse,random_search_best_params,optuna_best_rmse,optuna_best_params
0,Dataset_0136,Si Particle Size (um),xgboost,0.167731,{'model__learning_rate': np.float64(0.03142064...,0.156177,"{'model__n_estimators': 681, 'model__learning_..."
1,Dataset_0136,Si Particle Size (um),gpr,0.259406,{'model__alpha': np.float64(0.0580496585534013...,0.257787,{'model__alpha': 0.07789901985338313}
2,Dataset_0136,Si Particle Size (um),lightgbm,0.182430,{'model__learning_rate': np.float64(0.08124173...,0.184138,"{'model__n_estimators': 297, 'model__learning_..."
3,Dataset_0136,Hardness (HV),xgboost,1.077698,{'model__learning_rate': np.float64(0.07665788...,0.972185,"{'model__n_estimators': 173, 'model__learning_..."
4,Dataset_0136,Hardness (HV),gpr,1.458536,{'model__alpha': np.float64(1.1737996092661853...,1.458536,{'model__alpha': 1.6550213716969952e-05}
5,Dataset_0136,Hardness (HV),lightgbm,0.900878,{'model__learning_rate': np.float64(0.25540068...,0.819172,"{'model__n_estimators': 596, 'model__learning_..."
6,Dataset_0136,wear rate,xgboost,0.200207,{'model__learning_rate': np.float64(0.06416866...,0.177737,"{'model__n_estimators': 715, 'model__learning_..."
7,Dataset_0136,wear rate,gpr,0.290832,{'model__alpha': np.float64(3.197999889429474e...,0.290832,{'model__alpha': 3.284959743501089e-05}
8,Dataset_0136,wear rate,lightgbm,0.198334,{'model__learning_rate': np.float64(0.03385139...,0.206369,"{'model__n_estimators': 583, 'model__learning_..."
9,Dataset_0136,Ultimate Compression Strength (MPa),xgboost,3.933567,{'model__learning_rate': np.float64(0.07665788...,3.721806,"{'model__n_estimators': 697, 'model__learning_..."



Optuna improved on Random Search in 31 / 45 model/target/dataset combinations

Tuning results saved to: C:\Users\mohammadhosein\Desktop\DeepLearning-miniProject\results\tuning
